# MindTrace AI+ Advanced Emotion Model Training
This notebook trains an advanced Convolutional Neural Network on the FER-2013 dataset. It uses **Class Weights** to completely eliminate the "Sadness/Joy bias" and employs a much deeper architecture (VGG-style) for significantly higher accuracy.

**Instructions:**
1. Go to `Runtime` -> `Change runtime type` and ensure Hardware Accelerator is set to `GPU` (T4 is fine).
2. Upload your `fer2013.zip` dataset to the sidebar.
3. Run all cells (`Runtime` -> `Run all`).

In [ ]:
!pip install tensorflowjs
!unzip -q fer2013.zip -d fer2013_dataset

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization, Activation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

DATASET_DIR = "fer2013_dataset/fer2013"
IMG_SIZE = 48
BATCH_SIZE = 64
EPOCHS = 50
NUM_CLASSES = 7

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = test_datagen.flow_from_directory(
    os.path.join(DATASET_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# COMPUTE CLASS WEIGHTS TO FIX BIAS
# This forces the AI to pay 10x more attention to rare emotions like 'Disgust' and 'Surprise'
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes), 
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights_array))
print("Computed Class Weights to prevent bias:", class_weights)

# ADVANCED VGG-STYLE ARCHITECTURE
model = Sequential([
    # Block 1
    Conv2D(64, (3, 3), padding='same', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(64, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Block 2
    Conv2D(128, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(128, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Block 3
    Conv2D(256, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    Conv2D(256, (3, 3), padding='same'),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    # Fully Connected
    Flatten(),
    Dense(512),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

opt = Adam(learning_rate=0.001)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, verbose=1)
early_stop = EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=1)

print("\n--- STARTING UNBIASED TRAINING ---")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    class_weight=class_weights,  # FIXES THE BIAS
    callbacks=[reduce_lr, early_stop]
)

model.save("emotion_model.h5")

In [ ]:
!tensorflowjs_converter --input_format=keras emotion_model.h5 tfjs_model
!zip -r tfjs_model.zip tfjs_model

In [ ]:
from google.colab import files
files.download('tfjs_model.zip')